In [ ]:
"""
MASTER MIS Builder (PG + SHG)

This single Python program will:
1) Read LokOS SHG Excel file (your uploaded SHG Details Report -LokOS...)
2) Generate a "Master MIS" Excel workbook with:
   - README
   - LokOS_to_Tables (field mapping)
   - PG_Manual_to_Tables (field mapping skeleton you can extend)
   - SHG_Master (cleaned SHG master extracted from LokOS)
   - SHG_Bank_Accounts (ranked 1..3 accounts extracted)
   - Review_Queue (rows with missing / ambiguous data)
   - Screen_List
   - API_List

It does NOT require database access; it produces the MASTER SHEET (Excel) locally.

Requirements:
  pip install pandas openpyxl

Usage:
  python build_master_sheet.py \
      --lokos "SHG Details Report -LokOS 12.06.25 (1) (1).xlsx" \
      --out "Master_MIS.xlsx"
"""

import argparse
import re
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import pandas as pd


# ----------------------------
# Helpers
# ----------------------------
def norm_text(x) -> str:
    """Normalize text for stable matching / cleaning."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    return s


def safe_int(x) -> Optional[int]:
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        s = str(x).strip()
        if s == "":
            return None
        return int(float(s))
    except Exception:
        return None


def safe_float(x) -> Optional[float]:
    try:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        s = str(x).strip()
        if s == "":
            return None
        return float(s)
    except Exception:
        return None


def safe_date(x) -> Optional[pd.Timestamp]:
    """Try to parse date; return pandas Timestamp or None."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (datetime, pd.Timestamp)):
        return pd.Timestamp(x).normalize()
    s = str(x).strip()
    if s == "":
        return None
    # try day-first parsing typical in India
    try:
        return pd.to_datetime(s, dayfirst=True, errors="coerce")
    except Exception:
        return None


def read_lokos_excel(lokos_path: str) -> pd.DataFrame:
    """
    LokOS file structure (as per your earlier sheet):
    Row 1: header labels
    Row 2..: data

    We read with pandas, then re-assign columns from the first row.
    """
    raw = pd.read_excel(lokos_path, header=None)
    # Find the header row: usually row 0, but if there are top notes, detect the first row with "SHG Code"
    header_row_idx = None
    for i in range(min(10, len(raw))):
        rowvals = [norm_text(v) for v in raw.iloc[i].tolist()]
        if any(v.upper() == "SHG CODE" for v in rowvals):
            header_row_idx = i
            break
    if header_row_idx is None:
        header_row_idx = 0

    header = raw.iloc[header_row_idx].tolist()
    df = raw.iloc[header_row_idx + 1 :].copy()
    df.columns = header
    df = df.reset_index(drop=True)

    # Rename fully blank/NaN column names if any
    cols = []
    for c in df.columns:
        if c is None or (isinstance(c, float) and pd.isna(c)) or norm_text(c) == "":
            cols.append("Raw Row Id")
        else:
            cols.append(norm_text(c))
    df.columns = cols

    return df


def build_lokos_to_tables_mapping(df_columns: List[str]) -> pd.DataFrame:
    """
    Create a mapping sheet for LokOS Excel columns -> target DB table.
    This is a practical mapping; adjust if you renamed columns in your DB.
    """
    bank_keywords = ("Account Number", "Account Type", "IFSC", "Branch", "Bank Name", "Opening Date")
    rows = []
    for c in df_columns:
        c_norm = norm_text(c)
        if not c_norm:
            continue
        if any(k in c_norm for k in bank_keywords) or c_norm.lower().startswith(("1st ", "2nd ", "3rd ")):
            target = "mis.shg_bank_account (rank 1/2/3)"
        else:
            target = "mis.shg"
        rows.append([c_norm, target])
    return pd.DataFrame(rows, columns=["LokOS Excel Column", "Target Table"])


def build_pg_manual_mapping_skeleton() -> pd.DataFrame:
    """
    A usable mapping skeleton for PG manual -> DB. You can extend this sheet as needed.
    """
    rows = [
        ["District", "mis.pg.district_id (via mis.district)", "PG Registration"],
        ["Block", "mis.pg.block_id (via mis.block)", "PG Registration"],
        ["CLF", "mis.pg.clf_id (via mis.clf)", "PG Registration"],
        ["VO", "mis.pg.vo_id (via mis.vo)", "PG Registration"],
        ["GP/VC (LGD)", "mis.pg.gp_vc_id -> mis.gp_vc.lgd_code", "PG Registration"],
        ["Hamlet/Para", "mis.pg.hamlet_para_id", "PG Registration"],
        ["PG Name", "mis.pg.pg_name", "PG Registration"],
        ["PG Type", "mis.pg.pg_type", "PG Registration"],
        ["Formation Date", "mis.pg.formation_date", "PG Registration"],
        ["President Name/Contact", "mis.person + mis.pg.president_person_id", "PG Registration"],
        ["Secretary Name/Contact", "mis.person + mis.pg.secretary_person_id", "PG Registration"],
        ["PG Bank/Branch/Account", "mis.pg.bank_name / branch_name / account_number", "PG Registration"],
        ["Aggregation Centre Name/Address", "mis.pg.aggregation_centre_name_address", "PG Registration"],
        ["Total PG Members", "mis.pg.total_members_recorded", "PG Registration"],
        ["Member Name", "mis.person.full_name", "PG Member"],
        ["Member Category/Caste", "mis.person.category", "PG Member"],
        ["Member Spouse Name", "mis.person.spouse_name", "PG Member"],
        ["Member SHG Name", "mis.pg_member.shg_name_as_entered + mis.pg_member.shg_id", "PG Member"],
        ["Member Contact", "mis.person.contact_number", "PG Member"],
        ["Member Govt ID", "mis.person.govt_id_type / govt_id_number", "PG Member"],
        ["Member Bank/Branch/Account", "mis.pg_member.bank_name / branch_name / member_account_number", "PG Member"],
        ["Membership Fee", "mis.pg_member.membership_fee", "PG Member"],
        ["Designation in PG", "mis.pg_member.designation_in_pg", "PG Member"],
        ["PG Loan + monthly repayment", "mis.pg_loan + mis.pg_loan_monthly", "Loans"],
        ["Member Loan + monthly repayment", "mis.pg_member_loan + mis.pg_member_loan_monthly", "Loans"],
        ["Fund/Grant", "mis.pg_grant", "Fund/Grant"],
        ["Business monthly header", "mis.pg_business_month", "Business"],
        ["Business 6.1 member lines", "mis.pg_business_member_line", "Business"],
        ["Business 6.2 market lines", "mis.pg_business_market_line", "Business"],
        ["MPR aggregates", "mis.v_*_mpr_month (views)", "Reports"],
        ["Income–Expenditure", "Derived from business/loans/grants", "Reports"],
        ["Gradation", "mis.pg_gradation_quarter", "Reports"],
    ]
    return pd.DataFrame(rows, columns=["PG Manual Field/Section", "Target Table.Column", "Module"])


def extract_shg_master_and_banks(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Extracts:
      - SHG_Master table-like dataframe
      - SHG_Bank_Accounts (ranked 1..3) dataframe
      - Review_Queue dataframe for missing essentials

    We keep columns common in your LokOS file (robust to minor header differences).
    """
    # Identify likely core columns (fallback to empty string if missing)
    def col(name: str) -> str:
        # exact match
        if name in df.columns:
            return name
        # try case-insensitive
        for c in df.columns:
            if norm_text(c).lower() == name.lower():
                return c
        return name  # may not exist

    # Core fields (adjust if your excel uses different header spellings)
    core_cols = {
        "State": col("State"),
        "District": col("District"),
        "Block": col("Block"),
        "Gram Panchayat": col("Gram Panchayat"),
        "Village": col("Village"),
        "Primary Parent CBO (VO)": col("Primary Parent CBO (VO)"),
        "Secondary Parent CBO (CLF)": col("Secondary Parent CBO (CLF)"),
        "SHG Code": col("SHG Code"),
        "SHG Name": col("SHG Name"),
        "SHG NIC Code": col("SHG NIC Code"),
        "Date of Formation": col("Date of Formation"),
        "Active Members": col("Active Members"),
        "Promoted By": col("Promoted By"),
        "Promoters Name": col("Promoters Name"),
        "Co-option Status": col("Co-option Status"),
        "Co-option/Revival Date": col("Co-option/Revival Date"),
        "SHG Type": col("SHG Type"),
        "Special SHG Type": col("Special SHG Type"),
        "Meeting Frequency": col("Meeting Frequency"),
        "Savings Frequency": col("Savings Frequency"),
        "Savings Amount": col("Savings Amount"),
        "Tenure of Office Bearers (Months)": col("Tenure of Office Bearers (Months)"),
        "Primary Livelihoods": col("Primary Livelihoods"),
        "Secondary Livelihoods": col("Secondary Livelihoods"),
        "Tertiary Livelihoods": col("Tertiary Livelihoods"),
        "Approval Status": col("Approval Status"),
        "First Time Approval Date": col("First Time Approval Date"),
        "Status (Active/Inactive)": col("Status (Active/Inactive)"),
        "Inactive/Reject Reason": col("Inactive/Reject Reason"),
        "eBK ID": col("eBK ID"),
        "eBK Name": col("eBK Name"),
        "eBK Mobile No": col("eBK Mobile No"),
        "Migrated/LokOS": col("Migrated/LokOS"),
        "Raw Row Id": col("Raw Row Id"),
    }

    # Build SHG master
    shg_rows = []
    review_rows = []

    for idx, r in df.iterrows():
        shg_code = norm_text(r.get(core_cols["SHG Code"], None))
        shg_name = norm_text(r.get(core_cols["SHG Name"], None))

        # Create a review queue for missing essentials
        issues = []
        if not shg_code:
            issues.append("Missing SHG Code")
        if not shg_name:
            issues.append("Missing SHG Name")

        if issues:
            review_rows.append({
                "RowIndex": idx,
                "SHG Code": shg_code,
                "SHG Name": shg_name,
                "Issue": "; ".join(issues)
            })
            continue

        shg_rows.append({
            "State": norm_text(r.get(core_cols["State"], None)),
            "District": norm_text(r.get(core_cols["District"], None)),
            "Block": norm_text(r.get(core_cols["Block"], None)),
            "Gram Panchayat": norm_text(r.get(core_cols["Gram Panchayat"], None)),
            "Village": norm_text(r.get(core_cols["Village"], None)),
            "VO": norm_text(r.get(core_cols["Primary Parent CBO (VO)"], None)),
            "CLF": norm_text(r.get(core_cols["Secondary Parent CBO (CLF)"], None)),
            "SHG Code": shg_code,
            "SHG Name": shg_name,
            "SHG NIC Code": norm_text(r.get(core_cols["SHG NIC Code"], None)),
            "Formation Date": safe_date(r.get(core_cols["Date of Formation"], None)),
            "Active Members": safe_int(r.get(core_cols["Active Members"], None)),
            "Promoted By": norm_text(r.get(core_cols["Promoted By"], None)),
            "Promoters Name": norm_text(r.get(core_cols["Promoters Name"], None)),
            "Co-option Status": norm_text(r.get(core_cols["Co-option Status"], None)),
            "Co-option/Revival Date": safe_date(r.get(core_cols["Co-option/Revival Date"], None)),
            "SHG Type": norm_text(r.get(core_cols["SHG Type"], None)),
            "Special SHG Type": norm_text(r.get(core_cols["Special SHG Type"], None)),
            "Meeting Frequency": norm_text(r.get(core_cols["Meeting Frequency"], None)),
            "Savings Frequency": norm_text(r.get(core_cols["Savings Frequency"], None)),
            "Savings Amount": safe_float(r.get(core_cols["Savings Amount"], None)),
            "Tenure Office Bearers (Months)": safe_int(r.get(core_cols["Tenure of Office Bearers (Months)"], None)),
            "Primary Livelihoods": norm_text(r.get(core_cols["Primary Livelihoods"], None)),
            "Secondary Livelihoods": norm_text(r.get(core_cols["Secondary Livelihoods"], None)),
            "Tertiary Livelihoods": norm_text(r.get(core_cols["Tertiary Livelihoods"], None)),
            "Approval Status": norm_text(r.get(core_cols["Approval Status"], None)),
            "First Approval Date": safe_date(r.get(core_cols["First Time Approval Date"], None)),
            "Status Active/Inactive": norm_text(r.get(core_cols["Status (Active/Inactive)"], None)),
            "Inactive/Reject Reason": norm_text(r.get(core_cols["Inactive/Reject Reason"], None)),
            "eBK ID": norm_text(r.get(core_cols["eBK ID"], None)),
            "eBK Name": norm_text(r.get(core_cols["eBK Name"], None)),
            "eBK Mobile No": norm_text(r.get(core_cols["eBK Mobile No"], None)),
            "Migrated/LokOS": norm_text(r.get(core_cols["Migrated/LokOS"], None)),
            "Raw Row Id": norm_text(r.get(core_cols["Raw Row Id"], None)),
        })

    shg_master = pd.DataFrame(shg_rows)

    # Bank accounts extraction (rank 1..3)
    bank_rows = []
    for idx, r in df.iterrows():
        shg_code = norm_text(r.get(core_cols["SHG Code"], None))
        if not shg_code:
            continue

        for rank, prefix in [(1, "1st"), (2, "2nd"), (3, "3rd")]:
            acc_no = norm_text(r.get(f"{prefix} Account Number", None))
            acc_type = norm_text(r.get(f"{prefix} Account Type", None))
            ifsc = norm_text(r.get(f"{prefix} IFSC", None))
            branch = norm_text(r.get(f"{prefix} Branch Name", None))
            bank = norm_text(r.get(f"{prefix} Bank Name", None))
            open_date = safe_date(r.get(f"{prefix} Account Opening Date", None))

            if any([acc_no, acc_type, ifsc, branch, bank, open_date]):
                bank_rows.append({
                    "SHG Code": shg_code,
                    "Account Rank": rank,
                    "Account Number": acc_no,
                    "Account Type": acc_type,
                    "IFSC": ifsc,
                    "Branch Name": branch,
                    "Bank Name": bank,
                    "Account Opening Date": open_date,
                })

    bank_accounts = pd.DataFrame(bank_rows)
    review_queue = pd.DataFrame(review_rows)

    return shg_master, bank_accounts, review_queue


def build_screen_list() -> pd.DataFrame:
    rows = [
        ["Login", "All", "Authentication"],
        ["User Management + Scope", "State Admin", "Create users; assign district/block/clf scope"],
        ["Masters Setup", "State Admin", "Location + CLF/VO masters"],
        ["SHG Import", "State/District", "Upload LokOS excel, preview, commit; review queue"],
        ["SHG Registry", "All (scoped)", "Search SHGs, view details/bank accounts"],
        ["Create PG", "Block/CLF", "PG registration (one-time)"],
        ["Manage PG Members", "Block/CLF", "Add/edit members; link SHG"],
        ["PG Change Request", "Block/CLF", "Request edits to PG master"],
        ["State Authorization Queue", "State", "Approve/reject master changes"],
        ["Monthly Workspace (PG+Month)", "Block/CLF", "Navigate monthly entry, submit/close"],
        ["PG Loans", "Block/CLF", "Loan entry + monthly repayment snapshot"],
        ["Member Loans", "Block/CLF", "Member loan entry + monthly snapshot"],
        ["Fund/Grant", "Block/CLF", "Infra fund received + utilization note"],
        ["Business Monthly", "Block/CLF", "6.1/6.2/6.3/6.4/6.5"],
        ["Dashboards (State/District/Block/CLF)", "Role-based", "KPIs + drilldowns"],
        ["MPR Reports", "Role-based", "PG + aggregates"],
        ["Income–Expenditure (FY)", "Role-based", "Auto computed statement"],
        ["Gradation (Quarterly)", "Role-based", "Enter metrics, compute grade"],
    ]
    return pd.DataFrame(rows, columns=["Screen", "Role", "Purpose"])


def build_api_list() -> pd.DataFrame:
    rows = [
        ["POST", "/api/v1/auth/login", "Login"],
        ["GET", "/api/v1/auth/me", "Session info"],
        ["POST", "/api/v1/users", "Create user"],
        ["POST", "/api/v1/users/{userId}/scope", "Assign scope"],
        ["POST", "/api/v1/shg/import", "Dry-run import (parse + validate)"],
        ["POST", "/api/v1/shg/import/commit", "Commit import"],
        ["GET", "/api/v1/shg", "List/filter SHGs"],
        ["GET", "/api/v1/shg/{shgId}", "SHG detail"],
        ["POST", "/api/v1/pg", "Create PG"],
        ["GET", "/api/v1/pg", "List PGs"],
        ["POST", "/api/v1/pg/{pgId}/members", "Add member"],
        ["POST", "/api/v1/pg/{pgId}/change-requests", "Request PG master edit"],
        ["POST", "/api/v1/pg/change-requests/{id}/approve", "Approve change request"],
        ["POST", "/api/v1/pg/{pgId}/business-month", "Upsert business header"],
        ["POST", "/api/v1/business-month/{id}/member-lines", "Upsert 6.1 lines"],
        ["POST", "/api/v1/business-month/{id}/market-lines", "Upsert 6.2 lines"],
        ["GET", "/api/v1/reports/mpr/block/{blockId}?monthKey=YYYY-MM-01", "Block MPR"],
        ["GET", "/api/v1/dashboard/block/{blockId}?monthKey=YYYY-MM-01", "Block dashboard"],
    ]
    return pd.DataFrame(rows, columns=["Method", "Endpoint", "Description"])


def write_excel(
    out_path: str,
    shg_master: pd.DataFrame,
    bank_accounts: pd.DataFrame,
    review_queue: pd.DataFrame,
    lokos_mapping: pd.DataFrame,
    pg_mapping: pd.DataFrame,
    screens: pd.DataFrame,
    apis: pd.DataFrame,
):
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        # README
        readme = pd.DataFrame(
            [
                ["MASTER MIS (PG + SHG) - Master Sheet"],
                ["Generated by build_master_sheet.py"],
                [f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"],
                [""],
                ["Sheets included:"],
                ["- SHG_Master: cleaned SHG registry from LokOS"],
                ["- SHG_Bank_Accounts: bank accounts rank 1..3"],
                ["- Review_Queue: rows that need review (missing code/name etc.)"],
                ["- LokOS_to_Tables: Excel->DB mapping"],
                ["- PG_Manual_to_Tables: PG manual->DB mapping skeleton"],
                ["- Screens, APIs"],
            ],
            columns=["README"],
        )
        readme.to_excel(writer, sheet_name="README", index=False)

        # Data sheets
        shg_master.to_excel(writer, sheet_name="SHG_Master", index=False)
        bank_accounts.to_excel(writer, sheet_name="SHG_Bank_Accounts", index=False)
        review_queue.to_excel(writer, sheet_name="Review_Queue", index=False)

        # Mapping sheets
        lokos_mapping.to_excel(writer, sheet_name="LokOS_to_Tables", index=False)
        pg_mapping.to_excel(writer, sheet_name="PG_Manual_to_Tables", index=False)

        # Screens & APIs
        screens.to_excel(writer, sheet_name="Screens", index=False)
        apis.to_excel(writer, sheet_name="APIs", index=False)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--lokos", required=True, help="Path to LokOS SHG Details Excel (.xlsx)")
    ap.add_argument("--out", default="Master_MIS.xlsx", help="Output Excel file name")
    args = ap.parse_args()

    df = read_lokos_excel(args.lokos)

    shg_master, bank_accounts, review_queue = extract_shg_master_and_banks(df)
    lokos_mapping = build_lokos_to_tables_mapping(list(df.columns))
    pg_mapping = build_pg_manual_mapping_skeleton()
    screens = build_screen_list()
    apis = build_api_list()

    write_excel(
        out_path=args.out,
        shg_master=shg_master,
        bank_accounts=bank_accounts,
        review_queue=review_queue,
        lokos_mapping=lokos_mapping,
        pg_mapping=pg_mapping,
        screens=screens,
        apis=apis,
    )

    print(f"✅ Master MIS Excel generated: {args.out}")
    print(f"   SHG rows: {len(shg_master)}")
    print(f"   Bank rows: {len(bank_accounts)}")
    print(f"   Review rows: {len(review_queue)}")


if __name__ == "__main__":
    main()
